[CAPSTONE INTEGRATION TARGET]
    Learn the unions, cuts and intersections used to drill bracket holes and hollow the sensor housing. Analytic volume identities make material removal testable.


# 1. Overlap has a measurable meaning
    Both boxes have equal width and height. Their X overlap is a length in millimetres. The browser demonstration shows their 2D footprint; the native recipe verifies actual 3D solids.


In [ ]:
from __future__ import annotations
import importlib.util
import json
import math
import os
from pathlib import Path
import sys

if sys.version_info < (3, 12):
    raise RuntimeError("Choose the CortexCAD Python 3.12+ Jupyter kernel")

# Installed packages work from any directory. A source checkout can be selected
# explicitly; ancestor discovery also supports launching Jupyter inside a phase.
if importlib.util.find_spec("cortexcad") is None:
    root_hint: str = os.environ.get("CORTEXCAD_ROOT", "")
    candidates: list[Path] = ([Path(root_hint).expanduser()] if root_hint else [])
    candidates += [Path.cwd(), *Path.cwd().parents]
    source_root: Path | None = next((path for path in candidates
                                    if (path / "cortexcad" / "recipes.py").is_file()), None)
    if source_root is None:
        raise RuntimeError("Install CortexCAD in this kernel or set CORTEXCAD_ROOT to its checkout")
    sys.path.insert(0, str(source_root.resolve()))

from cortexcad.client import FreeCADClient
from cortexcad.recipes import DEFAULTS, build_script, validate_parameters
LIVE: bool = os.environ.get("CORTEXCAD_LIVE") == "1"
print("LIVE execution enabled" if LIVE else "OFFLINE: no CAD connection will be opened")


In [ ]:
PHASE: str = "csg"
params: dict[str, float] = validate_parameters(PHASE, {"length": 40, "width": 30, "height": 12, "overlap": 15})
section: float = params["width"] * params["height"]
operand_volume: float = params["length"] * section
intersection_volume: float = params["overlap"] * section
union_volume: float = 2 * operand_volume - intersection_volume
difference_volume: float = operand_volume - intersection_volume
assert math.isclose(union_volume + intersection_volume, 2 * operand_volume)
assert math.isclose(difference_volume + intersection_volume, operand_volume)
print({"union_mm3": union_volume, "difference_mm3": difference_volume, "intersection_mm3": intersection_volume})


## 2. Preserve construction history
    The script creates named `Part::Box` operands and linked Fuse/Cut/Common features. It recomputes the document before inspecting shapes. Union is visible initially; toggle one result at a time in FreeCAD to avoid overlapping displays. Default expectations are union 23,400, difference 9,000, intersection 5,400 mm³.


In [ ]:
script: str = build_script(PHASE, params)
compile(script, "freecad_lesson.py", "exec")
print(script)
print("Compiled source only: no FreeCAD solver or solid has run on this host.")


In [ ]:
if LIVE:
    # Endpoint and explicit timeout come from CORTEXCAD_ENDPOINT/TIMEOUT.
    client: FreeCADClient = FreeCADClient(timeout=float(os.environ.get("CORTEXCAD_TIMEOUT", "5")))
    outcome = client.run_recipe(PHASE, params)
    print(repr(outcome))
    print("Inspect the workstation and native assertions before accepting this lesson.")
else:
    print("OFFLINE preview complete; native geometry and hardware-fit gates remain pending.")


## 3. Explore the domain before calling CAD
    A zero intersection and a complete overlap violate this lesson's contract. Reject them locally; the exercise requires all three results to remain nonempty single solids.


In [ ]:
for overlap in (5.0, 15.0, 35.0):
    candidate: dict[str, float] = validate_parameters(PHASE, {**params, "overlap": overlap})
    common: float = overlap * section
    print(overlap, {"union": 2 * operand_volume - common, "difference": operand_volume - common, "intersection": common})
for overlap in (0.0, params["length"]):
    try:
        build_script(PHASE, {**params, "overlap": overlap})
    except ValueError as error:
        print("Expected rejection:", error)
    else:
        raise AssertionError("Degenerate overlap accepted")


## 4. Acceptance and extension
    Record native validity, one-solid checks and volumes for all five features. Explain why the same-volume A−B and B−A results are not necessarily the same shape. Then sketch how a cylindrical drill differs from a translated rectangular operand: axis, radius and traversing depth must all be specified.

    Save accepted CAD evidence and the parameter revision. Native execution and physical fit remain pending when this notebook runs offline.
